# Original PCBert-Kla training to KlaDB external study

This experiment compares PCBert-Kla (`baseline`), MM-Kla (`token_gated`), and architecture-aligned HybridKla using the original PCBert-Kla training source and the same locked KlaDB human study (PMID 41192556). A candidate-protein/homology-group-disjoint validation subset is formed within the original source for epoch selection. PCBert-Kla and MM-Kla are refit on **all 3,487 original rows** before external inference. HybridKla uses grouped out-of-fold stacking on the **grouped training subset** and is not refit on all original rows; its training budget and data use differ. The results table labels that distinction. This run does not use the external study for fitting, epoch selection, or threshold tuning.

The original training data contain repeated windows and candidate rather than verified parent-protein identities. External negative labels mean unreported lysines, not experimentally established non-lactylated sites. This same KlaDB holdout has been examined in earlier project experiments, so it is **not a newly blinded test**. HybridKla here is architecture-aligned retraining, not an exact reproduction of its authors' published training protocol.


## 1. Runtime and run control

Select a Kaggle **2 x T4 GPU** runtime with Internet enabled. Attach the private Kaggle dataset containing `kladb_human_external_test.csv` and `external_test_manifest.json`, and create a Kaggle secret named `GITHUB_TOKEN` with read access to the private repository. For the complete three-model experiment, set both `RUN_TRAINING = True` and `RUN_HYBRIDKLA_FULL = True`; optionally run `RUN_HYBRIDKLA_PILOT = True` first in a separate version to estimate compute. All training switches are off by default. One seed can be run per saved version; multiple seeds can also be listed in `RUN_SEEDS`.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, time, urllib.request, zipfile

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_DIR = Path('/kaggle/working/MM-Kla')
PROJECT = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT = Path('/kaggle/working/original_train_kladb_external_results')
RUN_TRAINING = False
RUN_HYBRIDKLA_PILOT = False
RUN_HYBRIDKLA_FULL = False
LOCAL_HYBRIDKLA_ARCHIVE = None
RUN_SEEDS = (42,)
DECLARED_SEEDS = (42, 123, 777)
EPOCHS = 30
BATCH_SIZE = 4
if not RUN_SEEDS or len(set(RUN_SEEDS)) != len(RUN_SEEDS) or any(seed not in DECLARED_SEEDS for seed in RUN_SEEDS):
    raise ValueError('RUN_SEEDS must be distinct members of DECLARED_SEEDS')
OUTPUT.mkdir(parents=True, exist_ok=True)


## 2. Source checkout and dependencies

Only the tracked original training files are obtained from GitHub. The access token is passed to Git through a temporary askpass helper; it is not printed or stored in notebook outputs. The checked-out commit and GPU hardware are recorded alongside the results.


In [ ]:
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        result = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
        del token, env
    if result.returncode:
        raise RuntimeError('Private clone failed; check repository access and Kaggle Internet')
if not (PROJECT / 'scripts/run_original_kladb_external.py').is_file():
    raise RuntimeError('The checked-out repository does not contain this experiment runner; update the checkout')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU runtime is required')
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
runtime = {'git_commit': COMMIT, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'torch': torch.__version__, 'seeds': RUN_SEEDS, 'max_epochs': EPOCHS, 'batch_size': BATCH_SIZE}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


## 3. External dataset and preflight

The attached dataset must contain exactly one matching pair of export files. The runner checks its checksum and audited counts, the original source-file checksums, exact training/test window overlap, feature schema, and grouped validation separation. This cell performs **no model training and no external inference**.


In [ ]:
matches = sorted(Path('/kaggle/input').rglob('kladb_human_external_test.csv'))
if len(matches) != 1:
    raise RuntimeError(f'Attach exactly one private KlaDB external dataset; found {len(matches)} CSV files')
EXTERNAL_CSV = matches[0]
EXTERNAL_MANIFEST = EXTERNAL_CSV.with_name('external_test_manifest.json')
if not EXTERNAL_MANIFEST.is_file():
    raise FileNotFoundError(EXTERNAL_MANIFEST)
manifest = json.loads(EXTERNAL_MANIFEST.read_text())
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if manifest['sha256'] != sha256(EXTERNAL_CSV):
    raise ValueError('Attached external CSV does not match its manifest')
print('External dataset:', EXTERNAL_CSV.parent)
print('External CSV SHA-256:', manifest['sha256'])
preflight = subprocess.run([sys.executable, 'scripts/run_original_kladb_external.py', '--architecture', 'baseline', '--seed', str(RUN_SEEDS[0]), '--external-csv', str(EXTERNAL_CSV), '--external-manifest', str(EXTERNAL_MANIFEST), '--output-dir', str(OUTPUT / 'preflight_unused'), '--check-only'], cwd=PROJECT, text=True, capture_output=True)
if preflight.returncode:
    raise RuntimeError('Preflight failed:\n' + preflight.stdout[-3000:] + preflight.stderr[-3000:])
print(preflight.stdout)


## 4. Matched two-GPU training

For each selected seed, the two architectures run in separate processes, one per T4. Both use ProtBert (`Rostlab/prot_bert`, first four encoder layers fine-tuned), 27 sequence-derived physicochemical features, batch size 4, SGD with learning rate 0.003, no positive weighting, no scheduler, and a fixed 0.5 decision threshold. Each run saves grouped-validation history, a full-data refit checkpoint and scaler, external probabilities, final metrics, and its console log. Model selection uses validation AUPRC only.

Kaggle Save Version starts a fresh `/kaggle/working`; collect output bundles across saved versions before computing a multi-seed summary. Changing `RUN_SEEDS` does not alter the external dataset.


In [ ]:
def launch(architecture, seed, device, training_scope='full_refit'):
    destination = OUTPUT / f'seed_{seed}' / (architecture if training_scope == 'full_refit' else f'matched_subset/{architecture}')
    if (destination / 'run_report.json').exists():
        raise RuntimeError(f'Results already exist at {destination}; use a fresh Kaggle session or archive them')
    destination.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', 'scripts/run_original_kladb_external.py', '--architecture', architecture, '--seed', str(seed), '--device', device, '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--training-scope', training_scope, '--external-csv', str(EXTERNAL_CSV), '--external-manifest', str(EXTERNAL_MANIFEST), '--output-dir', str(destination)]
    log = (destination / 'training.log').open('w')
    return subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT), log, destination

if RUN_TRAINING:
    for seed in RUN_SEEDS:
        jobs = {}
        try:
            for architecture, device in (('baseline', 'cuda:0'), ('token_gated', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')):
                jobs[architecture] = launch(architecture, seed, device)
                if torch.cuda.device_count() == 1:
                    process, log, destination = jobs.pop(architecture)
                    code = process.wait()
                    log.close()
                    if code:
                        raise RuntimeError(f'{architecture} seed {seed} failed; inspect {destination / "training.log"}')
            while any(process.poll() is None for process, _, _ in jobs.values()):
                time.sleep(60)
                for architecture, (process, _, destination) in jobs.items():
                    lines = (destination / 'training.log').read_text(errors='replace').splitlines()
                    print(f'seed {seed}, {architecture}:', lines[-1] if lines else 'starting', flush=True)
            failures = []
            for architecture, (process, log, destination) in jobs.items():
                code = process.wait()
                log.close()
                if code:
                    failures.append(f'{architecture} (exit {code}; {destination / "training.log"})')
            jobs.clear()
            if failures:
                raise RuntimeError('Training failed: ' + ', '.join(failures))
            print('Completed matched seed', seed)
        finally:
            for process, log, _ in jobs.values():
                if process.poll() is None:
                    process.terminate()
                    process.wait()
                if not log.closed:
                    log.close()
else:
    print('Training disabled; set RUN_TRAINING = True after reviewing the preflight')


## 5. Matched three-model subset comparison

HybridKla has eight branches: fine-tuned ESM2, an LSTM, six sequence-derived feature networks, and a stacking classifier. Five candidate-homology-grouped folds generate out-of-fold scores. Validation uses only the original training source. The KlaDB study is evaluated after selection. This is **not an exact reproduction** of the authors' published training protocol. To compare all three methods on the same original-source training rows, enabling the full HybridKla run also trains a matched-subset PCBert-Kla and MM-Kla pair. These extra ProtBert runs retain their validation-selected checkpoints instead of refitting on all 3,487 rows. They are separate from the full-refit pair in section 4. The models still have different architectures and optimization budgets.

The official HybridKla feature-resource archive is checksum-verified. It can be downloaded or supplied through LOCAL_HYBRIDKLA_ARCHIVE. RUN_HYBRIDKLA_PILOT trains one ESM2 fold for one epoch without external inference. RUN_HYBRIDKLA_FULL runs the matched pair and all HybridKla branches, then evaluates the external study. It can take substantially longer than section 4.


In [ ]:
if RUN_HYBRIDKLA_PILOT or RUN_HYBRIDKLA_FULL:
    resource_root = OUTPUT / 'hybridkla_resources'
    resource_root.mkdir(parents=True, exist_ok=True)
    if LOCAL_HYBRIDKLA_ARCHIVE is None:
        archive_path = resource_root / 'HybridKla.zip'
        if not archive_path.is_file():
            urllib.request.urlretrieve('https://transkla.zzu.edu.cn/api/download/HybridKla.zip', archive_path)
    else:
        archive_path = Path(LOCAL_HYBRIDKLA_ARCHIVE)
    if not archive_path.is_file() or sha256(archive_path) != '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993':
        raise ValueError('Official HybridKla archive is missing or its checksum changed')
    resources = resource_root / 'HybridKla' / 'train'
    resources.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in ('top10.txt', 'BLOSUM62R.txt'):
            (resources / filename).write_bytes(archive.read('HybridKla/train/' + filename))
    if RUN_HYBRIDKLA_FULL:
        for seed in RUN_SEEDS:
            jobs = {}
            try:
                for architecture, device in (('baseline', 'cuda:0'), ('token_gated', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')):
                    jobs[architecture] = launch(architecture, seed, device, training_scope='grouped_subset')
                    if torch.cuda.device_count() == 1:
                        process, log, destination = jobs.pop(architecture)
                        code = process.wait()
                        log.close()
                        if code:
                            raise RuntimeError(f'Matched-subset {architecture} failed; see {destination / "training.log"}')
                while any(process.poll() is None for process, _, _ in jobs.values()):
                    time.sleep(60)
                    for architecture, (process, _, destination) in jobs.items():
                        lines = (destination / 'training.log').read_text(errors='replace').splitlines()
                        print(f'matched subset seed {seed}, {architecture}:', lines[-1] if lines else 'starting', flush=True)
                failures = []
                for architecture, (process, log, destination) in jobs.items():
                    code = process.wait()
                    log.close()
                    if code:
                        failures.append(f'{architecture} (exit {code}; {destination / "training.log"})')
                jobs.clear()
                if failures:
                    raise RuntimeError('Matched-subset training failed: ' + ', '.join(failures))
            finally:
                for process, log, _ in jobs.values():
                    if process.poll() is None:
                        process.terminate()
                        process.wait()
                    if not log.closed:
                        log.close()
    def run_hybridkla(seed, pilot):
        destination = OUTPUT / f'seed_{seed}' / ('hybridkla_pilot' if pilot else 'hybridkla_architecture_aligned')
        if (destination / 'run_report.json').exists():
            raise RuntimeError(f'HybridKla results already exist at {destination}')
        destination.mkdir(parents=True, exist_ok=True)
        command = [sys.executable, '-u', 'scripts/run_hybridkla_architecture.py', '--original-external-csv', str(EXTERNAL_CSV), '--original-external-manifest', str(EXTERNAL_MANIFEST), '--resource-dir', str(resources), '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(seed)]
        if pilot:
            command.append('--pilot')
        log = (destination / 'training.log').open('w')
        try:
            process = subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
            while process.poll() is None:
                time.sleep(60)
                lines = (destination / 'training.log').read_text(errors='replace').splitlines()
                print(f'HybridKla seed {seed}:', lines[-1] if lines else 'starting', flush=True)
            if process.returncode:
                raise RuntimeError(f'HybridKla failed; see {destination / "training.log"}')
        finally:
            if 'process' in locals() and process.poll() is None:
                process.terminate()
                process.wait()
            log.close()
        report = json.loads((destination / 'run_report.json').read_text())
        if report['seed'] != seed or report['benchmark_lock_sha256'] != sha256(EXTERNAL_MANIFEST) or report['benchmark_input_audit']['external_csv_sha256'] != manifest['sha256']:
            raise ValueError('HybridKla result provenance mismatch')
        if (pilot and report['test_metrics']) or (not pilot and 'external_test' not in report['test_metrics']):
            raise ValueError('HybridKla test-use boundary was not respected')
        return report
    if RUN_HYBRIDKLA_PILOT:
        pilot_report = run_hybridkla(RUN_SEEDS[0], pilot=True)
        print('HybridKla one-fold ESM2 pilot seconds:', pilot_report['fold_reports']['ESM2_fold_0']['wall_seconds'])
    if RUN_HYBRIDKLA_FULL:
        for seed in RUN_SEEDS:
            full_report = run_hybridkla(seed, pilot=False)
            print('HybridKla external test, seed', seed, full_report['test_metrics']['external_test'])
else:
    print('HybridKla disabled; enable a pilot or full run in section 1')


## 6. Generated results and integrity checks

The tables below are built only from completed run reports. The full-refit PCBert-Kla/MM-Kla comparison and the matched-subset three-model comparison are kept **separate**. Every prediction file must cover the same 5,095 external sites with finite probabilities. The training_rows and training_protocol columns show the actual data use. No published metrics are inserted. A three-model mean/std table requires all methods and all declared seeds in the same Kaggle output directory.


In [ ]:
import pandas as pd
import numpy as np
locked_sites = pd.read_csv(EXTERNAL_CSV)
locked_names = locked_sites['accession'].astype(str) + ':' + locked_sites['position_1based'].astype(str)
records = []
split_hashes = {}
def add_protbert_result(seed, architecture, comparison_scope):
    directory = (OUTPUT / f'seed_{seed}' / architecture if comparison_scope == 'full_original_refit' else OUTPUT / f'seed_{seed}' / 'matched_subset' / architecture)
    report_file = directory / 'run_report.json'
    if not report_file.is_file():
        return
    report = json.loads(report_file.read_text())
    expected_training_scope = 'full_refit' if comparison_scope == 'full_original_refit' else 'grouped_subset'
    if report['seed'] != seed or report['architecture'] != architecture or report['training_scope'] != expected_training_scope or report['git_commit'] != COMMIT or report['preflight']['external_sha256'] != manifest['sha256']:
        raise ValueError(f'Result provenance mismatch for {architecture}, seed {seed}')
    predictions = pd.read_csv(directory / 'external_predictions.csv')
    if len(predictions) != 5095 or not predictions['name'].astype(str).equals(locked_names) or not predictions['true_label'].astype(int).equals(locked_sites['label'].astype(int)) or not predictions['window_51aa'].astype(str).equals(locked_sites['window_51aa'].astype(str)) or not np.isfinite(predictions['score']).all() or not predictions['score'].between(0, 1).all():
        raise ValueError(f'Invalid external predictions for {architecture}, seed {seed}')
    training_rows = report['refit_training']['training_positive_rows'] + report['refit_training']['training_negative_rows']
    if comparison_scope == 'matched_grouped_subset':
        split_hashes[(seed, architecture)] = (report['preflight']['validation']['training_indices_sha256'], report['preflight']['validation']['validation_indices_sha256'])
    records.append({'comparison_scope': comparison_scope, 'architecture': architecture, 'seed': seed, 'training_rows': training_rows, 'training_protocol': report['training_scope'], 'selected_epoch': report['selected_epoch'], **report['external_metrics']})
for seed in DECLARED_SEEDS:
    for architecture in ('baseline', 'token_gated'):
        add_protbert_result(seed, architecture, 'full_original_refit')
        add_protbert_result(seed, architecture, 'matched_grouped_subset')
    hybrid_dir = OUTPUT / f'seed_{seed}' / 'hybridkla_architecture_aligned'
    hybrid_report = hybrid_dir / 'run_report.json'
    if hybrid_report.is_file():
        report = json.loads(hybrid_report.read_text())
        audit = report['benchmark_input_audit']
        if report['seed'] != seed or report['git_commit'] != COMMIT or report['benchmark_lock_sha256'] != sha256(EXTERNAL_MANIFEST) or audit['external_csv_sha256'] != manifest['sha256'] or report['mode'] != 'locked-test-evaluation':
            raise ValueError(f'HybridKla provenance mismatch for seed {seed}')
        predictions = pd.read_csv(hybrid_dir / 'external_test_predictions.csv')
        if len(predictions) != 5095 or not predictions['Site_ID'].astype(str).equals(locked_names) or not predictions['Label'].astype(int).equals(locked_sites['label'].astype(int)) or not np.isfinite(predictions['Score']).all() or not predictions['Score'].between(0, 1).all():
            raise ValueError(f'Invalid HybridKla external predictions for seed {seed}')
        split_hashes[(seed, 'hybridkla_architecture_aligned')] = (audit['training_indices_sha256'], audit['validation_indices_sha256'])
        records.append({'comparison_scope': 'matched_grouped_subset', 'architecture': 'hybridkla_architecture_aligned', 'seed': seed, 'training_rows': audit['training_rows'], 'training_protocol': audit['training_scope'], 'selected_epoch': report['fold_reports']['META']['selected_epoch'], **report['test_metrics']['external_test']})
for seed in DECLARED_SEEDS:
    hashes = [value for (run_seed, _), value in split_hashes.items() if run_seed == seed]
    if len(set(hashes)) > 1:
        raise ValueError(f'Three-model training/validation partitions differ for seed {seed}')
results = pd.DataFrame.from_records(records)
if results.empty:
    print('No completed external evaluations in this Kaggle output directory')
else:
    display(results)
    results.to_csv(OUTPUT / 'external_results.csv', index=False)
    matched = results.loc[results['comparison_scope'] == 'matched_grouped_subset']
    expected = {(seed, architecture) for seed in DECLARED_SEEDS for architecture in ('baseline', 'token_gated', 'hybridkla_architecture_aligned')}
    if set(zip(matched['seed'], matched['architecture'])) == expected and len(matched) == len(expected):
        summary = matched.groupby('architecture')[['AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC']].agg(['mean', 'std'])
        display(summary)
        summary.to_csv(OUTPUT / 'matched_three_model_summary.csv')
    else:
        print('Matched three-model, three-seed summary pending; combine output bundles first')


## Interpretation boundary

Compare AUPRC against the external positive prevalence (254/5,095), and report AUC, MCC, sensitivity, and specificity with the stated 0.5 threshold. This protocol may yield lower scores than the original PCBert-Kla balanced test, which had substantial exact-window overlap with training. The old training duplicates remain a limitation, and the candidate-protein homology screen does not prove every original record's parent identity. Because this KlaDB holdout has already been examined in earlier project work, report this evaluation as **previously inspected external-study data**, not a fresh blinded confirmatory test. Further model changes would require a new untouched test for a confirmatory claim.
